# E-Commerce Customer Behavior - Exploratory Data Analysis (EDA)

This notebook explores the ecommerce dataset with 8000 users and analyzes the `ad_clicked` target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported")

## 1. Load Data

In [ ]:
# Load raw data
data_path = Path('../data/ecommerce_user_behavior_8000.csv')
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")
print(f"\n💾 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2. Data Overview

In [ ]:
# First few rows
print("🔍 First 5 rows:")
print(df.head())

print("\n📊 Data Types:")
print(df.dtypes)

print("\n📈 Statistical Summary:")
print(df.describe())

## 3. Missing Values Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percent': missing_pct.values
}).sort_values('Missing_Count', ascending=False)

print("❌ Missing Values:")
print(missing_df[missing_df['Missing_Count'] > 0])

# Visualize
if missing.sum() > 0:
    missing_df_sorted = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count')
    plt.figure(figsize=(10, 6))
    plt.barh(missing_df_sorted['Column'], missing_df_sorted['Missing_Percent'], color='coral')
    plt.xlabel('Missing Percentage (%)')
    plt.title('Missing Values Distribution')
    plt.tight_layout()
    plt.savefig('../figures/01_missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✅ Saved: figures/01_missing_values.png")

## 4. Target Variable Analysis (ad_clicked)

In [ ]:
# Target distribution
print("🎯 Target Variable (ad_clicked):")
print(f"Total: {len(df)}")
print(f"\nValue counts (with NaN):")
print(df['ad_clicked'].value_counts(dropna=False))

print(f"\nPercentage:")
print(df['ad_clicked'].value_counts(normalize=True, dropna=False) * 100)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df_clean = df[df['ad_clicked'].notna()]
df_clean['ad_clicked'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Ad Clicked Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Ad Clicked')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No', 'Yes'], rotation=0)

# Pie chart
df_clean['ad_clicked'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                             labels=['Yes', 'No'], colors=['#4ECDC4', '#FF6B6B'])
axes[1].set_title('Ad Clicked Ratio', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../figures/02_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Saved: figures/02_target_distribution.png")

## 5. Feature Analysis

In [ ]:
# Numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('user_id')  # Remove ID

print("🔢 Numerical Features:")
print(numerical_cols)

# Categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\n🏷️ Categorical Features:")
print(categorical_cols)

### 5.1 Numerical Features Distribution

In [ ]:
# Distribution of numerical features
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols[:12]):
    axes[idx].hist(df[col].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col}', fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/03_numerical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/03_numerical_distributions.png")

### 5.2 Categorical Features

In [ ]:
# Categorical features
for col in categorical_cols:
    print(f"\n📊 {col}:")
    print(df[col].value_counts(dropna=False))

### 5.3 Feature vs Target Analysis

In [ ]:
# Features vs Target
df_clean = df[df['ad_clicked'].notna()]

# For numerical features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

features_to_plot = ['age', 'time_on_site', 'pages_viewed', 'previous_purchases', 'avg_session_time', 'bounce_rate']

for idx, col in enumerate(features_to_plot):
    if col in df_clean.columns:
        df_clean.boxplot(column=col, by='ad_clicked', ax=axes[idx])
        axes[idx].set_title(f'{col} vs Ad Clicked', fontweight='bold')
        axes[idx].set_xlabel('Ad Clicked')
        axes[idx].set_ylabel(col)
        plt.sca(axes[idx])
        plt.xticks([1, 2], ['No', 'Yes'])

plt.suptitle('', fontsize=1)  # Remove automatic title
plt.tight_layout()
plt.savefig('../figures/04_features_vs_target.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/04_features_vs_target.png")

### 5.4 Categorical Features vs Target

In [ ]:
# Categorical features vs target
for col in categorical_cols:
    print(f"\n🔗 {col} vs ad_clicked:")
    ct = pd.crosstab(df_clean[col], df_clean['ad_clicked'], margins=True)
    print(ct)
    print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender vs Ad Clicked
pd.crosstab(df_clean['gender'], df_clean['ad_clicked']).plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Gender vs Ad Clicked', fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(['No', 'Yes'])
plt.sca(axes[0])
plt.xticks(rotation=45)

# Device Type vs Ad Clicked
pd.crosstab(df_clean['device_type'], df_clean['ad_clicked']).plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'])
axes[1].set_title('Device Type vs Ad Clicked', fontweight='bold')
axes[1].set_xlabel('Device Type')
axes[1].set_ylabel('Count')
axes[1].legend(['No', 'Yes'])
plt.sca(axes[1])
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('../figures/05_categorical_vs_target.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/05_categorical_vs_target.png")

## 6. Correlation Analysis

In [ ]:
# Correlation with target
correlations = df_clean.corr()['ad_clicked'].sort_values(ascending=False)

print("📊 Correlation with ad_clicked:")
print(correlations)

# Correlation heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df_clean.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            cbar_kws={'label': 'Correlation'}, square=True)
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/06_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Saved: figures/06_correlation_matrix.png")

## 7. Key Insights Summary

In [ ]:
print("""
📊 KEY FINDINGS FROM EDA
========================

1. DATASET SIZE:
   - Total Records: {}
   - Missing Target: {}
   - Usable Records: {}

2. TARGET BALANCE:
   - Ad Clicked: {} ({:.1f}%)
   - Ad Not Clicked: {} ({:.1f}%)

3. TOP PREDICTIVE FEATURES (by correlation):
""".format(
    len(df),
    df['ad_clicked'].isnull().sum(),
    len(df_clean),
    (df_clean['ad_clicked'] == 1).sum(),
    (df_clean['ad_clicked'] == 1).sum() / len(df_clean) * 100,
    (df_clean['ad_clicked'] == 0).sum(),
    (df_clean['ad_clicked'] == 0).sum() / len(df_clean) * 100
))

top_corr = correlations.drop('ad_clicked').abs().nlargest(5)
for i, (col, val) in enumerate(top_corr.items(), 1):
    print(f"   {i}. {col}: {correlations[col]:.3f}")

print("""
4. DATA QUALITY:
   - Nearly Balanced Classes (Good for modeling)
   - Some Missing Values in {} columns
   - Mixed Numerical & Categorical Features

5. NEXT STEPS:
   ✓ Handle missing values
   ✓ Encode categorical variables
   ✓ Train Random Forest classifier
   ✓ Evaluate model performance
""".format(len(missing_df[missing_df['Missing_Count'] > 0])))

---

**Analysis Complete!** ✅

Generated files:
- `figures/01_missing_values.png`
- `figures/02_target_distribution.png`
- `figures/03_numerical_distributions.png`
- `figures/04_features_vs_target.png`
- `figures/05_categorical_vs_target.png`
- `figures/06_correlation_matrix.png`